# Stage 5 — Temperature scaling for the locked ResNet18

This notebook calibrates the already selected Stage 4 model. It does not
retrain the classifier or revisit model selection.

The fixed protocol is: verify the split and checkpoint; generate validation
logits with the exact Stage 4 inference path; fit one positive scalar
temperature using validation NLL only; freeze it; and then evaluate calibrated
probabilities on test.

**Locked checkpoint SHA-256:**
`3f6a701aca082fa675ba229ddfbae0139346e3cbff84ea02ff0617ab73c5d657`

**Fixed split checksum:** `f97c4ec9a27435a932662d5a8b707255`


## 1. Runtime and reproducibility controls
   
On Kaggle, attach the Stage 4 checkpoint as a private dataset before running. Internet access is required to download EuroSAT and the committed split manifests.

In [ ]:
import hashlib
import json
import math
import platform
import random
import shutil
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
import torch
import torch.nn as nn
import torchvision
from PIL import Image
from scipy.optimize import minimize_scalar
from sklearn.metrics import accuracy_score, f1_score, log_loss
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import EuroSAT
from torchvision.models import ResNet18_Weights, resnet18
from tqdm.auto import tqdm

SEED = 42
EXPECTED_SPLIT_CHECKSUM = "f97c4ec9a27435a932662d5a8b707255"
EXPECTED_CHECKPOINT_SHA256 = (
    "3f6a701aca082fa675ba229ddfbae0139346e3cbff84ea02ff0617ab73c5d657"
)
NUM_CLASSES = 10
BATCH_SIZE = 64
ECE_BINS = 15
BOOTSTRAP_REPLICATES = 2_000


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


seed_everything()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = Path("/kaggle/working/results/temperature_scaling")
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

environment = {
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "torchvision": torchvision.__version__,
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "scikit_learn": sklearn.__version__,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "cuda": torch.version.cuda,
    "seed": SEED,
    "mixed_precision_inference": DEVICE.type == "cuda",
}
print(json.dumps(environment, indent=2))


## 2. Locate and verify the locked checkpoint

If automatic discovery finds more than one checkpoint, set CHECKPOINT_PATH explicitly to the correct Kaggle input path.

In [ ]:
# Example explicit value:
# CHECKPOINT_PATH = Path("/kaggle/input/calibrated-eurosat-stage4/candidate_a_best.pth")
CHECKPOINT_PATH = None


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


if CHECKPOINT_PATH is None:
    checkpoint_candidates = sorted(
        Path("/kaggle/input").rglob("candidate_a_best.pth")
    )
    if len(checkpoint_candidates) != 1:
        raise RuntimeError(
            "Expected exactly one candidate_a_best.pth under /kaggle/input; "
            f"found {len(checkpoint_candidates)}: {checkpoint_candidates}. "
            "Set CHECKPOINT_PATH explicitly."
        )
    CHECKPOINT_PATH = checkpoint_candidates[0]

observed_checkpoint_sha256 = sha256_file(CHECKPOINT_PATH)
print("Checkpoint:", CHECKPOINT_PATH)
print("Observed SHA-256:", observed_checkpoint_sha256)
assert observed_checkpoint_sha256 == EXPECTED_CHECKPOINT_SHA256


## 3. Download EuroSAT and verify the fixed manifests
   
The original 70/15/15 split is reused without modification. No new split is generated.

In [ ]:
DATA_BASE = Path("/kaggle/working/data")
eurosat = EuroSAT(root=DATA_BASE, download=True)
DATA_ROOT = DATA_BASE / "eurosat" / "2750"

SPLIT_URL = (
    "https://raw.githubusercontent.com/"
    "Cricdatahater/calibrated-eurosat/main/data/splits"
)
splits = {
    name: pd.read_csv(f"{SPLIT_URL}/{name}.csv")
    for name in ("train", "validation", "test")
}
expected_sizes = {"train": 18_900, "validation": 4_050, "test": 4_050}

split_index_bytes = urllib.request.urlopen(
    f"{SPLIT_URL}/split_indices.json"
).read()
observed_split_checksum = hashlib.md5(split_index_bytes).hexdigest()
assert observed_split_checksum == EXPECTED_SPLIT_CHECKSUM

all_indices = pd.concat(
    [frame["dataset_index"] for frame in splits.values()], ignore_index=True
)
assert len(all_indices) == 27_000
assert all_indices.nunique() == 27_000
for name, size in expected_sizes.items():
    assert len(splits[name]) == size

expected_mapping = (
    pd.concat(splits.values(), ignore_index=True)
    [["class_name", "class_index"]]
    .drop_duplicates()
    .set_index("class_name")["class_index"]
    .to_dict()
)
assert eurosat.class_to_idx == expected_mapping
CLASS_NAMES = eurosat.classes

print("Split checksum:", observed_split_checksum)
print({name: len(frame) for name, frame in splits.items()})
print("Classes:", CLASS_NAMES)


## 4. Deterministic evaluation datasets
   
Validation and test images use the preprocessing bundled with IMAGENET1K_V1, exactly as in Stage 4.

In [ ]:
weights = ResNet18_Weights.IMAGENET1K_V1
evaluation_transform = weights.transforms()


class ManifestDataset(Dataset):
    def __init__(self, manifest, data_root, transform):
        self.manifest = manifest.reset_index(drop=True)
        self.data_root = Path(data_root)
        self.transform = transform

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, index):
        row = self.manifest.iloc[index]
        image_path = (
            self.data_root
            / row["class_name"]
            / Path(row["relative_path"]).name
        )
        with Image.open(image_path) as image:
            image = self.transform(image.convert("RGB"))
        return image, int(row["class_index"]), int(row["dataset_index"])


validation_dataset = ManifestDataset(
    splits["validation"], DATA_ROOT, evaluation_transform
)
test_dataset = ManifestDataset(splits["test"], DATA_ROOT, evaluation_transform)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

assert len(validation_dataset) == 4_050
assert len(test_dataset) == 4_050


## 5. Restore the locked Stage 4 model


In [ ]:
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)
assert checkpoint["candidate"] == "candidate_a"
assert checkpoint["epoch"] == 12
assert checkpoint["split_checksum"] == EXPECTED_SPLIT_CHECKSUM

model = resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(DEVICE)
model.eval()

for parameter in model.parameters():
    parameter.requires_grad = False

print("Loaded locked Candidate A, epoch 12.")


## 6. Generate validation logits

Only validation outputs are used to estimate the temperature.

In [ ]:
def collect_logits(model, loader, description):
    logits_parts = []
    label_parts = []
    index_parts = []

    model.eval()
    with torch.inference_mode():
        for images, labels, indices in tqdm(loader, desc=description):
            images = images.to(DEVICE, non_blocking=True)
            # Stage 4 used CUDA autocast. Reusing it here preserves the locked
            # model's archived validation and test predictions.
            with torch.amp.autocast(
                device_type=DEVICE.type,
                enabled=DEVICE.type == "cuda",
            ):
                batch_logits = model(images)
            logits_parts.append(batch_logits.float().cpu())
            label_parts.append(labels.cpu())
            index_parts.append(indices.cpu())

    logits = torch.cat(logits_parts).numpy()
    labels = torch.cat(label_parts).numpy().astype(np.int64)
    indices = torch.cat(index_parts).numpy().astype(np.int64)

    assert logits.shape == (len(loader.dataset), NUM_CLASSES)
    assert len(np.unique(indices)) == len(indices)
    assert np.isfinite(logits).all()
    return logits, labels, indices


validation_logits, y_validation, validation_indices = collect_logits(
    model, validation_loader, "Validation inference"
)

np.savez_compressed(
    OUTPUT_DIR / "validation_logits.npz",
    logits=validation_logits,
    labels=y_validation,
    dataset_indices=validation_indices,
    class_names=np.asarray(CLASS_NAMES),
)
print("Validation logits:", validation_logits.shape)


## 7. Fit and freeze one scalar temperature

The optimized variable is log(T), ensuring T > 0. The objective is validation multiclass negative log-likelihood. The broad bounds prevent pathological numerical values while remaining non-informative for ordinary temperatures.

In [ ]:
def softmax_numpy(logits):
    shifted = logits - logits.max(axis=1, keepdims=True)
    exponentiated = np.exp(shifted)
    return exponentiated / exponentiated.sum(axis=1, keepdims=True)


def nll_at_log_temperature(log_temperature, logits, labels):
    temperature = math.exp(float(log_temperature))
    probabilities = softmax_numpy(logits / temperature)
    return log_loss(
        labels,
        probabilities,
        labels=list(range(NUM_CLASSES)),
    )


uncalibrated_validation_probabilities = softmax_numpy(validation_logits)
uncalibrated_validation_nll = log_loss(
    y_validation,
    uncalibrated_validation_probabilities,
    labels=list(range(NUM_CLASSES)),
)

optimization = minimize_scalar(
    nll_at_log_temperature,
    args=(validation_logits, y_validation),
    method="bounded",
    bounds=(math.log(0.05), math.log(10.0)),
    options={"xatol": 1e-12, "maxiter": 1_000},
)
assert optimization.success, optimization.message

TEMPERATURE = float(math.exp(optimization.x))
calibrated_validation_probabilities = softmax_numpy(
    validation_logits / TEMPERATURE
)
calibrated_validation_nll = log_loss(
    y_validation,
    calibrated_validation_probabilities,
    labels=list(range(NUM_CLASSES)),
)

assert TEMPERATURE > 0
assert calibrated_validation_nll <= uncalibrated_validation_nll + 1e-12

temperature_record = {
    "temperature": TEMPERATURE,
    "parameterization": "single_positive_scalar",
    "fit_partition": "validation",
    "objective": "multiclass_negative_log_likelihood",
    "optimizer": "scipy.optimize.minimize_scalar_bounded_on_log_temperature",
    "validation_samples": len(y_validation),
    "validation_nll_before": uncalibrated_validation_nll,
    "validation_nll_after": calibrated_validation_nll,
    "checkpoint_sha256": observed_checkpoint_sha256,
    "split_checksum": observed_split_checksum,
}
with (OUTPUT_DIR / "temperature.json").open("w") as file:
    json.dump(temperature_record, file, indent=2)

print(json.dumps(temperature_record, indent=2))


## 8. Locked test evaluation

The temperature is now frozen. The following cells do not alter it or any part of the classifier.

In [ ]:
test_logits, y_test, test_indices = collect_logits(
    model, test_loader, "Locked test inference"
)
np.savez_compressed(
    OUTPUT_DIR / "test_logits.npz",
    logits=test_logits,
    labels=y_test,
    dataset_indices=test_indices,
    class_names=np.asarray(CLASS_NAMES),
)

uncalibrated_test_probabilities = softmax_numpy(test_logits)
calibrated_test_probabilities = softmax_numpy(test_logits / TEMPERATURE)

uncalibrated_predictions = uncalibrated_test_probabilities.argmax(axis=1)
calibrated_predictions = calibrated_test_probabilities.argmax(axis=1)

# A positive scalar temperature must preserve every class prediction.
assert np.array_equal(uncalibrated_predictions, calibrated_predictions)
print("All test class predictions are unchanged.")


## 9. Metrics and reliability bins


In [ ]:
def multiclass_brier_score(y_true, probabilities):
    one_hot = np.eye(probabilities.shape[1])[y_true]
    return float(np.mean(np.sum((probabilities - one_hot) ** 2, axis=1)))


def calculate_ece(y_true, probabilities, n_bins=ECE_BINS):
    predictions = probabilities.argmax(axis=1)
    confidence = probabilities.max(axis=1)
    correct = predictions == y_true
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    records = []
    ece = 0.0

    for bin_index, (lower, upper) in enumerate(zip(edges[:-1], edges[1:])):
        if bin_index == 0:
            mask = (confidence >= lower) & (confidence <= upper)
        else:
            mask = (confidence > lower) & (confidence <= upper)
        count = int(mask.sum())
        accuracy = float(correct[mask].mean()) if count else np.nan
        mean_confidence = float(confidence[mask].mean()) if count else np.nan
        if count:
            ece += (count / len(y_true)) * abs(accuracy - mean_confidence)
        records.append({
            "bin": bin_index + 1,
            "lower": lower,
            "upper": upper,
            "count": count,
            "accuracy": accuracy,
            "mean_confidence": mean_confidence,
        })
    return float(ece), pd.DataFrame(records)


def probability_metrics(y_true, probabilities):
    predictions = probabilities.argmax(axis=1)
    ece, bins = calculate_ece(y_true, probabilities)
    metrics = {
        "accuracy": float(accuracy_score(y_true, predictions)),
        "macro_f1": float(f1_score(y_true, predictions, average="macro")),
        "log_loss": float(log_loss(
            y_true, probabilities, labels=list(range(NUM_CLASSES))
        )),
        "multiclass_brier_score": multiclass_brier_score(
            y_true, probabilities
        ),
        "expected_calibration_error": ece,
    }
    return metrics, bins


validation_before, _ = probability_metrics(
    y_validation, uncalibrated_validation_probabilities
)
validation_after, _ = probability_metrics(
    y_validation, calibrated_validation_probabilities
)
test_before, test_bins_before = probability_metrics(
    y_test, uncalibrated_test_probabilities
)
test_after, test_bins_after = probability_metrics(
    y_test, calibrated_test_probabilities
)

metric_comparison = pd.DataFrame([
    {"partition": "validation", "calibration": "before", **validation_before},
    {"partition": "validation", "calibration": "after", **validation_after},
    {"partition": "test", "calibration": "before", **test_before},
    {"partition": "test", "calibration": "after", **test_after},
])
display(metric_comparison)

assert test_before["accuracy"] == test_after["accuracy"]
assert test_before["macro_f1"] == test_after["macro_f1"]

# These must reproduce the locked Stage 4 evaluation. A failure usually means
# the preprocessing, batch size, checkpoint, or mixed-precision path changed.
EXPECTED_STAGE4_TEST_ACCURACY = 0.9795061728395061
EXPECTED_STAGE4_TEST_MACRO_F1 = 0.9785400949175062
assert np.isclose(
    test_before["accuracy"], EXPECTED_STAGE4_TEST_ACCURACY, atol=1e-12
)
assert np.isclose(
    test_before["macro_f1"], EXPECTED_STAGE4_TEST_MACRO_F1, atol=1e-12
)

metric_comparison.to_csv(OUTPUT_DIR / "metric_comparison.csv", index=False)
test_bins_before.to_csv(
    OUTPUT_DIR / "test_calibration_bins_uncalibrated.csv", index=False
)
test_bins_after.to_csv(
    OUTPUT_DIR / "test_calibration_bins_calibrated.csv", index=False
)


## 10. Paired bootstrap confidence intervals

Bootstrap differences are defined as calibrated − uncalibrated; negative differences favor calibration for all three probability metrics. ECE intervals should be interpreted cautiously because ECE depends on the chosen bins.

In [ ]:
def metric_triplet(labels, probabilities):
    ece, _ = calculate_ece(labels, probabilities)
    return np.array([
        log_loss(labels, probabilities, labels=list(range(NUM_CLASSES))),
        multiclass_brier_score(labels, probabilities),
        ece,
    ])


rng = np.random.default_rng(SEED)
bootstrap_differences = np.empty((BOOTSTRAP_REPLICATES, 3), dtype=float)
n_test = len(y_test)

for replicate in tqdm(range(BOOTSTRAP_REPLICATES), desc="Paired bootstrap"):
    sample = rng.integers(0, n_test, size=n_test)
    labels = y_test[sample]
    before = metric_triplet(labels, uncalibrated_test_probabilities[sample])
    after = metric_triplet(labels, calibrated_test_probabilities[sample])
    bootstrap_differences[replicate] = after - before

metric_names = [
    "log_loss",
    "multiclass_brier_score",
    "expected_calibration_error",
]
point_differences = (
    metric_triplet(y_test, calibrated_test_probabilities)
    - metric_triplet(y_test, uncalibrated_test_probabilities)
)

bootstrap_intervals = pd.DataFrame({
    "metric": metric_names,
    "difference_calibrated_minus_uncalibrated": point_differences,
    "ci_2.5_percent": np.quantile(bootstrap_differences, 0.025, axis=0),
    "ci_97.5_percent": np.quantile(bootstrap_differences, 0.975, axis=0),
    "bootstrap_replicates": BOOTSTRAP_REPLICATES,
})
bootstrap_intervals.to_csv(OUTPUT_DIR / "bootstrap_intervals.csv", index=False)
display(bootstrap_intervals)


## 11. Reliability and confidence figures


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for bins, label, color in [
    (test_bins_before, "Uncalibrated", "tab:blue"),
    (test_bins_after, "Temperature-scaled", "tab:orange"),
]:
    nonempty = bins["count"] > 0
    axes[0].plot(
        bins.loc[nonempty, "mean_confidence"],
        bins.loc[nonempty, "accuracy"],
        marker="o",
        label=label,
        color=color,
    )
axes[0].plot([0, 1], [0, 1], "--", color="black", linewidth=1)
axes[0].set(xlabel="Mean confidence", ylabel="Accuracy", title="Test reliability")
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)
axes[0].legend()

axes[1].hist(
    uncalibrated_test_probabilities.max(axis=1),
    bins=np.linspace(0, 1, 21),
    alpha=0.55,
    label="Uncalibrated",
)
axes[1].hist(
    calibrated_test_probabilities.max(axis=1),
    bins=np.linspace(0, 1, 21),
    alpha=0.55,
    label="Temperature-scaled",
)
axes[1].set(xlabel="Top-label confidence", ylabel="Images", title="Test confidence")
axes[1].legend()

fig.tight_layout()
fig.savefig(FIGURE_DIR / "reliability_and_confidence.svg", bbox_inches="tight")
plt.show()


## 12. Export predictions and summary


In [ ]:
prediction_table = pd.DataFrame({
    "dataset_index": test_indices,
    "true_class_index": y_test,
    "true_class_name": [CLASS_NAMES[index] for index in y_test],
    "predicted_class_index": calibrated_predictions,
    "predicted_class_name": [CLASS_NAMES[index] for index in calibrated_predictions],
    "confidence_uncalibrated": uncalibrated_test_probabilities.max(axis=1),
    "confidence_calibrated": calibrated_test_probabilities.max(axis=1),
})
for class_index, class_name in enumerate(CLASS_NAMES):
    prediction_table[f"probability_uncalibrated_{class_name}"] = (
        uncalibrated_test_probabilities[:, class_index]
    )
    prediction_table[f"probability_calibrated_{class_name}"] = (
        calibrated_test_probabilities[:, class_index]
    )
prediction_table.to_csv(
    OUTPUT_DIR / "test_predictions_calibrated.csv", index=False
)

summary = {
    "stage": 5,
    "method": "single_scalar_temperature_scaling",
    "temperature": TEMPERATURE,
    "fit_partition": "validation_only",
    "checkpoint_sha256": observed_checkpoint_sha256,
    "split_checksum": observed_split_checksum,
    "classification_predictions_unchanged": bool(
        np.array_equal(uncalibrated_predictions, calibrated_predictions)
    ),
    "validation_metrics_before": validation_before,
    "validation_metrics_after": validation_after,
    "test_metrics_before": test_before,
    "test_metrics_after": test_after,
    "bootstrap_difference_definition": "calibrated_minus_uncalibrated",
    "bootstrap_replicates": BOOTSTRAP_REPLICATES,
    "environment": environment,
    "limitations": [
        "ECE depends on the selected 15-bin definition.",
        "Temperature was estimated on one fixed validation partition.",
        "The image-level split does not establish geographic generalization.",
        "Test results were not used to select or alter the temperature.",
    ],
}
with (OUTPUT_DIR / "calibration_summary.json").open("w") as file:
    json.dump(summary, file, indent=2)
with (OUTPUT_DIR / "environment.json").open("w") as file:
    json.dump(environment, file, indent=2)

archive_path = shutil.make_archive(
    "/kaggle/working/stage5_temperature_scaling_artifacts",
    "zip",
    root_dir=OUTPUT_DIR,
)
print(json.dumps(summary, indent=2))
print("Artifact archive:", archive_path)


## 13. Interpretation checklist

- Report the fitted temperature and validation NLL change.
- Treat lower test log loss and Brier score as the primary evidence of improved probability quality.
- Report ECE even if it moves in the opposite direction; do not tune the temperature against test ECE.
- State that accuracy and macro-F1 are unchanged by construction.
- Preserve the geographic-generalization limitation.
- Download the artifact ZIP, clean this notebook, and commit reproducible non-checkpoint outputs. The checkpoint remains Git-ignored.